In [61]:
schemal = {
  "city": "Lagos",
  "country": "Nigeria",
  "daily_weather": {
    "monday": {
      "temperature": 70,
      "weather": "sunny",
      "humidity": 50,
      "wind": 10
    },
    "tuesday": {
      "temperature": 65,
      "weather": "cloudy",
      "humidity": 60,
      "wind": 5
    },
    "wednesday": {
      "temperature": 60,
      "weather": "rainy",
      "humidity": 70,
      "wind": 15
    },
    "thursday": {
      "temperature": 55,
      "weather": "cloudy",
      "humidity": 80,
      "wind": 10
    },
    "friday": {
      "temperature": 50,
      "weather": "rainy",
      "humidity": 90,
      "wind": 15
    },
    "saturday": {
      "temperature": 45,
      "weather": "cloudy",
      "humidity": 100,
      "wind": 20
    },
    "sunday": {
      "temperature": 40,
      "weather": "rainy",
      "humidity": 100,
      "wind": 20
    }
  }
}

In [62]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    raise ValueError("OPENAI_API_KEY is not set")
print("OPENAI_API_KEY is set")

OLLAMA_URL = "http://localhost:11434/v1"

MODEL_CONFIG = {
    "llama3.2": {"base_url": OLLAMA_URL, "api_key": "ollama", "model": "llama3.2", "temperature": 0.7},
    "gpt-oss": {"base_url": OLLAMA_URL, "api_key": "ollama", "model": "gpt-oss", "temperature": 0.7},
    "gpt-5-nano": {"base_url": None, "api_key": OPENAI_API_KEY, "model": "gpt-5-nano", "temperature": 1},
}

CITIES = [
    "Lagos", "New York", "London", "Toronto", "Berlin",
    "Paris", "Tokyo", "Sydney", "São Paulo", "Mumbai",
    "Cape Town", "Mexico City", "Beijing", "Rome", "Madrid",
    "Nairobi", "Accra", "Cairo", "Buenos Aires", "Seoul",
]

OPENAI_API_KEY is set


In [63]:
TEMP_UNITS = {"°F": "Fahrenheit (°F)", "°C": "Celsius (°C)"}

SYSTEM_PROMPT_TEMPLATE = """\
You are a weather data generator. Generate realistic synthetic weather data as a JSON object.
Temperatures MUST be in {unit_label}.
The response MUST follow this exact structure:
{{
  "city": "<the requested city>",
  "country": "<country the city belongs to>",
  "daily_weather": {{
    "monday":    {{"temperature": <{unit_symbol}>, "weather": "<condition>", "humidity": <0-100>, "wind": <mph>}},
    "tuesday":   {{"temperature": <{unit_symbol}>, "weather": "<condition>", "humidity": <0-100>, "wind": <mph>}},
    "wednesday": {{"temperature": <{unit_symbol}>, "weather": "<condition>", "humidity": <0-100>, "wind": <mph>}},
    "thursday":  {{"temperature": <{unit_symbol}>, "weather": "<condition>", "humidity": <0-100>, "wind": <mph>}},
    "friday":    {{"temperature": <{unit_symbol}>, "weather": "<condition>", "humidity": <0-100>, "wind": <mph>}},
    "saturday":  {{"temperature": <{unit_symbol}>, "weather": "<condition>", "humidity": <0-100>, "wind": <mph>}},
    "sunday":    {{"temperature": <{unit_symbol}>, "weather": "<condition>", "humidity": <0-100>, "wind": <mph>}}
  }}
}}
<condition> must be one of: sunny, cloudy, rainy, stormy, snowy, windy, foggy.
Generate data that is climatically realistic for the city's climate and current season.
Return ONLY the JSON object."""


def _build_client(model_name: str) -> OpenAI:
    cfg = MODEL_CONFIG[model_name]
    kwargs = {"api_key": cfg["api_key"]}
    if cfg["base_url"]:
        kwargs["base_url"] = cfg["base_url"]
    return OpenAI(**kwargs)


def generate_weather(city: str, model_name: str, temp_unit: str) -> tuple[str, str]:
    unit_symbol = temp_unit
    unit_label = TEMP_UNITS[temp_unit]
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(unit_symbol=unit_symbol, unit_label=unit_label)

    try:
        client = _build_client(model_name)
        response = client.chat.completions.create(
            model=MODEL_CONFIG[model_name]["model"],
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Generate a 7-day weather forecast for {city}."},
            ],
            response_format={"type": "json_object"},
            temperature=MODEL_CONFIG[model_name]["temperature"],
        )

        data = json.loads(response.choices[0].message.content)
        temps = [day["temperature"] for day in data["daily_weather"].values()]

        metadata = (
            f"City:          {data['city']}\n"
            f"Country:       {data['country']}\n"
            f"Lowest Temp:   {min(temps)}{unit_symbol}\n"
            f"Highest Temp:  {max(temps)}{unit_symbol}"
        )
        return metadata, json.dumps(data, indent=2)

    except Exception as exc:
        error_json = json.dumps({"error": str(exc)}, indent=2)
        return f"Error: {exc}", error_json

In [64]:
with gr.Blocks(
    title="Weather Data Generator",
    theme=gr.themes.Soft(
        primary_hue=gr.themes.Color(
            c50="#e8f4ff", c100="#d0e8ff", c200="#a1d1ff", c300="#72baff",
            c400="#43a3ff", c500="#1e90ff", c600="#1a7de0", c700="#156abf",
            c800="#10579f", c900="#0b4480", c950="#083366",
        ),
    ),
    css="""
        .gradio-container .block,
        .gradio-container .wrap,
        .gradio-container .panel,
        .gradio-container .form,
        .gradio-container .block.padded,
        .gradio-container .border_focus,
        .gradio-container .border,
        .gradio-container .container > div { border-radius: 20px !important; }
        .gradio-container textarea,
        .gradio-container input,
        .gradio-container select,
        .gradio-container button,
        .gradio-container .code-block,
        .gradio-container .input-container,
        .gradio-container [class*="textbox"],
        .gradio-container [class*="dropdown"],
        .gradio-container [data-testid="textbox"],
        .gradio-container [data-testid="dropdown"] { border-radius: 20px !important; }
        .gradio-container .secondary-wrap,
        .gradio-container .primary-wrap { border-radius: 20px !important; }
        .gradio-container span[data-testid="block-info"],
        .gradio-container .block > div { border-radius: 20px !important; }
        #json-output, #json-output > * { max-height: 520px; overflow-y: auto; border-radius: 20px !important; }
    """,
) as demo:
    gr.Markdown("# Synthetic Weather Data Generator")
    gr.Markdown("Pick a city and a model, then hit **Generate** to create a realistic 7-day weather forecast.")

    with gr.Row(equal_height=True):
        # ── left column: inputs ──
        with gr.Column(scale=1, min_width=260):
            city_dd = gr.Dropdown(
                choices=CITIES,
                value="Lagos",
                label="City",
            )
            model_dd = gr.Dropdown(
                choices=list(MODEL_CONFIG.keys()),
                value="llama3.2",
                label="Model",
            )
            temp_unit_dd = gr.Dropdown(
                choices=list(TEMP_UNITS.keys()),
                value="°F",
                label="Temperature Unit",
            )
            generate_btn = gr.Button("Generate", variant="primary", size="lg")

        # ── right column: outputs ──
        with gr.Column(scale=2):
            metadata_box = gr.Textbox(
                label="Summary",
                lines=4,
                max_lines=4,
                interactive=False,
            )
            json_output = gr.Code(
                label="Weather Data (JSON)",
                language="json",
                lines=25,
                elem_id="json-output",
            )

    generate_btn.click(
        fn=generate_weather,
        inputs=[city_dd, model_dd, temp_unit_dd],
        outputs=[metadata_box, json_output],
    )

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.
